# Step 13 · Generate host × pathogen-class feature-question drafts

Use GPT-6.1 Terra to propose a complete, pair-contextual candidate question set for every distinct host in `configs/host_choices.yaml` crossed with every pathogen class in `configs/pathogen_taxonomies.yaml`. Existing names that share an NCBI taxon ID are treated as aliases, so each taxon appears once.

The scientific objective is to assess whether each pathogen causes disease in the host, then characterize pathogen properties and pathogen–host interactions that matter to this pathogenicity. Exclude generic biological facts that do not illuminate the disease-causing relationship. The model returns a concise human-readable label and definition, a complete feature question, scope, and applicability traits. This preliminary version promotes all inherited and generated questions for use. Experts can remove questions or add new ones in a later catalogue/profile version while retaining this snapshot unchanged. Each question also receives the full Jev request template used by V4: a `noul` typed yes/no question with the canonical true/false criteria and state placeholders for pathogen, host, title, and abstract. The prompt receives the curated host and pathogen traits recorded for each matrix cell. This notebook writes a separate draft YAML and resumable per-cell checkpoints under `data/v4/question_generation/`; it does not overwrite the production feature catalogue. Review scientific accuracy, abstract observability, overlap, pathogenicity relevance, and trait assignments before integrating any candidates.

A durable checkpoint is stored per host × pathogen-class cell; it is independent of catalogue version, so versioning and selection do not trigger duplicate model calls. Before generation, current resolved feature questions are exported into the same checkpoint schema for combinations without a checkpoint. Completed LLM or catalogue-export checkpoints are then reused, so only genuinely missing combinations call the API. Set `OPENAI_API_KEY` before running. The default model is `gpt-6.1-terra`; change `MODEL` below if needed.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pathogen_atlas_v4").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pathogen_atlas_v4").is_dir():
    raise RuntimeError("Open this notebook from the Pathogen Atlas V4 repository")

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


## 1 · Load the canonical host, pathogen-class, and trait registries

In [ ]:
import hashlib
import json
import re
from datetime import datetime, timezone
from itertools import product
from pathogen_atlas_v4.host_taxonomy.ncbi import PILOT_HOSTS
from pathogen_atlas_v4.features import resolve_profile

import yaml
from IPython.display import display
import pandas as pd

HOSTS_PATH = PROJECT_ROOT / "configs/host_choices.yaml"
PATHOGENS_PATH = PROJECT_ROOT / "configs/pathogen_taxonomies.yaml"
FEATURES_PATH = PROJECT_ROOT / "configs/features.yaml"

with HOSTS_PATH.open(encoding="utf-8") as f:
    host_choices = yaml.safe_load(f)
with PATHOGENS_PATH.open(encoding="utf-8") as f:
    pathogen_registry = yaml.safe_load(f)
with FEATURES_PATH.open(encoding="utf-8") as f:
    feature_catalogue = yaml.safe_load(f)

# Keep the first configured display name for each taxon; the remaining names are aliases.
host_aliases_by_id = {}
for name, taxon_id in host_choices["host_name_to_id"].items():
    host_aliases_by_id.setdefault(str(taxon_id), []).append(str(name))
hosts = [
    {"host_id": taxon_id, "taxon_id": taxon_id.removeprefix("ncbi:"),
     "label": names[0], "aliases": names[1:],
     "host_group": PILOT_HOSTS.get(taxon_id.removeprefix("ncbi:"), "other")}
    for taxon_id, names in host_aliases_by_id.items()
]
pathogen_groups = [
    {"pathogen_group": str(group_id), "label": str(group["label"])}
    for group_id, group in pathogen_registry["pathogen_groups"].items()
]

# Traits are an allow-list. This generator may assign these traits to question applicability,
# but it cannot mint biological traits for a host or pathogen.
trait_assignments = feature_catalogue.get("trait_assignments", {})
allowed_traits = {
    "host": set(map(str, feature_catalogue.get("traits", {}).get("host", []))),
    "pathogen": set(map(str, feature_catalogue.get("traits", {}).get("pathogen", []))),
    "pair": set(map(str, feature_catalogue.get("traits", {}).get("pair", []))),
}
for selector, trait_map in (("host_group", "host"), ("host_taxon", "host"),
                            ("pathogen_group", "pathogen"), ("pair_context", "pair")):
    for values in trait_assignments.get(selector, {}).values():
        allowed_traits[trait_map].update(map(str, values or []))

if not hosts or not pathogen_groups:
    raise ValueError("Host or pathogen-class registry is empty")
if any(not h["host_id"] for h in hosts):
    raise ValueError("Every host must have a taxon ID")

matrix = list(product(hosts, pathogen_groups))
print(f"Distinct hosts: {len(hosts)} | pathogen classes: {len(pathogen_groups)} | pair profiles: {len(matrix)}")
display(pd.DataFrame([
    {"host": host["label"], "host_id": host["host_id"],
     "pathogen_class": pathogen["label"], "pathogen_group": pathogen["pathogen_group"]}
    for host, pathogen in matrix
]))


## 2 · Define the structured output and prompt

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field
from openai import OpenAI
from pathogen_atlas_v4.prompts import build_feature_question

JEV_PROMPTS_PATH = PROJECT_ROOT / "configs/prompts.yaml"
with JEV_PROMPTS_PATH.open(encoding="utf-8") as f:
    jev_prompt_catalogue = yaml.safe_load(f)

MODEL = "gpt-6.1-terra"
REASONING_EFFORT = "medium"
PROMPT_VERSION = "host-pathogen-pathogenicity-feature-question-draft-v2"
RELATIONSHIP_CONTEXT = "disease"

class CandidateQuestion(BaseModel):
    model_config = ConfigDict(extra="forbid")
    feature_id: str = Field(min_length=3, max_length=100, description="Stable snake_case ID unique within this draft profile")
    question_version: str = Field(default="llm-candidate-v1", min_length=1, max_length=100)
    dimension: str = Field(min_length=2, max_length=100)
    category: str = Field(min_length=2, max_length=100)
    label: str = Field(min_length=3, max_length=160)
    definition: str = Field(min_length=10, max_length=500)
    question: str = Field(min_length=15, max_length=1200)
    question_claim: str = Field(min_length=10, max_length=500)
    scope: Literal["pathogen", "host", "pathogen_host_pair"]
    applicability_host_traits: list[str] = Field(default_factory=list)
    applicability_pathogen_traits: list[str] = Field(default_factory=list)
    applicability_pair_traits: list[str] = Field(default_factory=list)
    abstract_observability_rationale: str = Field(min_length=10, max_length=500)
    ambiguity_or_review_note: str = Field(min_length=1, max_length=500)

class PairQuestionDraft(BaseModel):
    model_config = ConfigDict(extra="forbid")
    profile_summary: str = Field(min_length=10, max_length=1000)
    questions: list[CandidateQuestion] = Field(min_length=1, max_length=60)

SYSTEM_PROMPT = """You are designing a scientific feature-question catalogue for a pathogen–host disease evidence atlas. The Atlas's central objective is to assess whether a named pathogen causes disease in a named host. Your features must characterize the pathogen and its interaction or relationship with that host specifically with respect to pathogenicity: how the pathogen causes, presents, spreads, persists in, or affects the host in the disease context.

Generate a thorough, nonredundant set of candidate questions for the supplied host × pathogen-class profile. Every question must help characterize this disease-causing relationship or a pathogen property relevant to its ability to cause disease in this host. Do not generate generic host facts, generic organism facts, or general epidemiology that does not illuminate pathogenicity in the selected host. Do not treat mere detection, colonization, carriage, contamination, vector/reservoir status, experimental use, or co-occurrence as proof that the pathogen causes disease in the host. Avoid questions that merely repeat the separate disease-causation gate; instead ask about distinct pathogenic aspects that an abstract can explicitly report.

Questions will later be answered from article titles and abstracts. Make each question a single observable claim that can be explicitly supported or left unsupported by an abstract. Do not ask for general-knowledge inference or assume that an unmentioned property is biologically absent. Relevant dimensions may include pathogen virulence or pathogenic properties, mechanisms explicitly reported, host response or susceptibility in the disease context, tissues or organs involved, clinical or plant disease manifestations, transmission or exposure leading to disease, persistence, and host impact. Include a dimension only when it helps characterize pathogenicity for this particular host × pathogen-class combination. Do not force every category if abstracts are unlikely to report it. Separate bundled claims into separate questions. Avoid duplicates and overly broad questions. Aim for a useful comprehensive profile, usually 8–20 questions.

For every question, identify the narrowest valid scope: pathogen, host, or pathogen_host_pair. Even pathogen-scoped or host-scoped questions must be relevant to understanding the supplied disease-causing relationship. Provide one atomic positive claim. Write `question` in the full human-readable feature format: a complete, self-contained yes/no question Jev can use as `feature_question`, explicitly scoped to the named entity or pair and requiring support from the supplied abstract. The title may identify an article but cannot alone support a positive. A negative means only that this abstract does not support the claim.

Assign applicability traits only from the allowed vocabulary supplied in the request. Use empty lists when a question has no trait restriction on an axis. Do not invent traits, infer additional biological traits for the host or pathogen, or claim the supplied traits are complete. Pair traits may only use the supplied pair-context traits. IDs must be concise snake_case and unique inside this profile. Mark uncertainties or likely low-observability areas for human review. Return candidate questions, not asserted biological facts."""


## 3 · Set checkpoint and versioned output locations

`DRAFT_CATALOGUE_VERSION` and `DRAFT_PROFILE_VERSION` identify an immutable candidate snapshot. Increment them for a new snapshot. The source catalogue version is recorded as the parent. If the destination already exists, the notebook refuses to overwrite it.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data/v4/question_generation"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DRAFTS_DIR = OUTPUT_DIR / "drafts"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DRAFTS_DIR.mkdir(parents=True, exist_ok=True)

# Increment both values to create a new immutable candidate snapshot.
DRAFT_CATALOGUE_VERSION = "v4-feature-catalogue-2.0.2"
DRAFT_PROFILE_VERSION = "v4-feature-profile-2.0.2"
DRAFT_PATH = DRAFTS_DIR / f"{DRAFT_CATALOGUE_VERSION}.yaml"
DRAFT_ALREADY_EXISTS = DRAFT_PATH.exists()
if DRAFT_ALREADY_EXISTS:
    print(f"Immutable draft already exists and will not be overwritten: {DRAFT_PATH}")


def slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.casefold()).strip("-")


def assigned_traits_for_pair(host: dict, pathogen_group: str) -> dict:
    """Return only taxon/group traits already curated in the feature catalogue."""
    host_traits = set(map(str, trait_assignments.get("host_group", {}).get(host["host_group"], [])))
    host_id = host["host_id"]
    for selector, key in (("host_taxon", host_id), ("host_taxon", host_id.removeprefix("ncbi:"))):
        host_traits.update(map(str, trait_assignments.get(selector, {}).get(key, [])))
    pathogen_traits = set(map(str, trait_assignments.get("pathogen_group", {}).get(pathogen_group, [])))
    pair_traits = set(map(str, trait_assignments.get("pair_context", {}).get(RELATIONSHIP_CONTEXT, [])))
    return {"host": sorted(host_traits), "pathogen": sorted(pathogen_traits), "pair": sorted(pair_traits)}

def pair_payload(host: dict, pathogen: dict) -> dict:
    selectors = {
        "host_id": host["host_id"], "taxon_id": host["taxon_id"],
        "host_label": host["label"], "host_aliases": host["aliases"],
        "host_group": host["host_group"],
        "pathogen_group": pathogen["pathogen_group"], "pathogen_class_label": pathogen["label"],
        "relationship_context": RELATIONSHIP_CONTEXT,
    }
    return {
        "atlas_objective": (
            "Assess whether this pathogen causes disease in this host, then characterize pathogen properties "
            "and pathogen–host relationships that matter to that pathogenicity. Every generated feature must "
            "inform the disease-causing relationship, not generic biology."
        ),
        "selectors": selectors,
        "curated_traits": assigned_traits_for_pair(host, pathogen["pathogen_group"]),
        "allowed_applicability_traits": {axis: sorted(values) for axis, values in allowed_traits.items()},
        "design_constraints": {
            "evidence_source": "article title and abstract",
            "positive_rule": "explicit report for the named entity or pair",
            "negative_rule": "not supported by this abstract; not biological absence",
            "desired_question_count": "comprehensive but nonredundant, typically 8–20",
            "relevance_test": "Would this feature help characterize how this pathogen causes disease in this host? Exclude it if not.",
            "not_the_disease_gate": "Do not repeat the overall cause-of-disease yes/no gate; ask about distinct, explicit pathogenic aspects.",
        },
    }



## 4 · Export the current catalogue, then generate or resume missing profiles

Each nonempty resolved profile is serialized into the per-combination checkpoint format, including the full Jev yes/no template. Existing catalogue checkpoints are refreshed only when the active catalogue version/profile is newer; completed model profiles are preserved unless their questions have since been integrated into the active catalogue. The LLM is called only when neither a completed checkpoint nor an active resolved profile exists.

In [ ]:
def profile_checkpoint(host: dict, pathogen: dict) -> Path:
    """One stable, durable cache entry per host × pathogen-class combination."""
    return CHECKPOINT_DIR / f"{slug(host['host_id'])}__{pathogen['pathogen_group']}.json"


REFRESH_CACHE = False  # Set True only when intentionally regenerating completed combinations.

def export_current_catalogue_cache(host: dict, pathogen: dict) -> bool:
    """Serialize the current resolved profile as a reusable pair checkpoint."""
    checkpoint = profile_checkpoint(host, pathogen)
    cached = json.loads(checkpoint.read_text(encoding="utf-8")) if checkpoint.exists() else None

    resolved = resolve_profile(
        feature_catalogue,
        pathogen_group=pathogen["pathogen_group"],
        host_group=host["host_group"],
        host_taxon_id=host["taxon_id"],
        relationship_context=RELATIONSHIP_CONTEXT,
        profile_version=feature_catalogue.get("profile_version", ""),
    )
    features = resolved.get("ordered_features", [])
    if not features:
        return False
    current_ids = {str(feature["feature_id"]) for feature in features}
    if cached is not None:
        cached_ids = {str(feature.get("feature_id")) for feature in cached.get("result", {}).get("questions", [])}
        same_catalogue = cached.get("source_catalogue_version") == feature_catalogue.get("catalogue_version")
        same_input = cached.get("input") == pair_payload(host, pathogen)
        if cached.get("cache_origin") == "existing_feature_catalogue" and same_catalogue and same_input and cached_ids == current_ids:
            return False
        if cached.get("cache_origin") == "llm_generation" and not cached_ids.issubset(current_ids):
            return False  # Preserve completed model work that has not been added to the active catalogue.

    questions = []
    for original in features:
        feature = dict(original)
        feature["answer_type"] = feature.get("answer_type", feature_catalogue.get("answer_type", "noul"))
        feature["question_version"] = feature.get("question_version") or "catalogue-v1"
        template = build_feature_question(
            pathogen_name="{pathogen.accepted_name}", pathogen_id="{pathogen.id}",
            host_name="{host.accepted_name}", host_id="{host.id}", feature=feature,
            article_title="{article.title}", article_abstract="{article.abstract}",
            prompt_config=jev_prompt_catalogue,
        )
        feature["jev_question_template"] = template
        feature["review_status"] = feature.get("review_status", "existing-catalogue")
        feature["selected"] = True
        questions.append(feature)

    input_payload = pair_payload(host, pathogen)
    previous_generation = None
    if cached is not None and cached.get("cache_origin") == "llm_generation":
        previous_generation = {
            "model": cached.get("model"),
            "prompt_version": cached.get("prompt_version"),
            "generated_at_utc": cached.get("generated_at_utc"),
            "system_prompt_sha256": cached.get("system_prompt_sha256"),
            "question_ids": sorted({str(q.get("feature_id")) for q in cached.get("result", {}).get("questions", [])}),
        }
    record = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "model": "existing-feature-catalogue",
        "prompt_version": "feature-catalogue-export-v1",
        "cache_origin": "existing_feature_catalogue",
        "source_catalogue_version": feature_catalogue.get("catalogue_version", ""),
        "source_profile_version": feature_catalogue.get("profile_version", ""),
        "previous_generation_cache": previous_generation,
        "input": input_payload,
        "result": {
            "selectors": input_payload["selectors"],
            "curated_traits": input_payload["curated_traits"],
            "profile_summary": f"Exported {len(questions)} current resolved questions from catalogue {feature_catalogue.get('catalogue_version', '')}.",
            "questions": questions,
        },
    }
    validate_question_records(questions)
    checkpoint.write_text(json.dumps(record, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return True


def validate_pair_draft(draft: PairQuestionDraft) -> PairQuestionDraft:
    seen = set()
    for question in draft.questions:
        if not re.fullmatch(r"[a-z][a-z0-9_]*", question.feature_id):
            raise ValueError(f"Invalid feature_id: {question.feature_id}")
        if question.feature_id in seen:
            raise ValueError(f"Duplicate feature_id: {question.feature_id}")
        seen.add(question.feature_id)
        for axis, values in (("host", question.applicability_host_traits),
                             ("pathogen", question.applicability_pathogen_traits),
                             ("pair", question.applicability_pair_traits)):
            unknown = sorted(set(values) - allowed_traits[axis])
            if unknown:
                raise ValueError(f"{question.feature_id} uses unknown {axis} traits: {unknown}")
    return draft


def validate_question_records(records: list[dict]) -> None:
    seen = set()
    for question in records:
        feature_id = str(question.get("feature_id", ""))
        if not re.fullmatch(r"[a-z][a-z0-9_]*", feature_id) or feature_id in seen:
            raise ValueError(f"Invalid or duplicate feature_id in cache: {feature_id!r}")
        seen.add(feature_id)
        if question.get("answer_type") != "noul":
            raise ValueError(f"{feature_id} is not configured as a Jev noul yes/no question")
        applicability = question.get("applicability", {})
        for axis, key in (("host", "host_traits"), ("pathogen", "pathogen_traits"), ("pair", "pair_traits")):
            rule = applicability.get(key, {})
            traits = set(map(str, rule.get("all_of", []))) | set(map(str, rule.get("any_of", [])))
            if traits - allowed_traits[axis]:
                raise ValueError(f"{feature_id} cache contains unregistered {axis} traits: {sorted(traits - allowed_traits[axis])}")
        template = question.get("jev_question_template", {})
        typed = template.get("questions", {}).get("feature", {})
        if typed.get("type") != "noul" or not {"true", "false"}.issubset(typed.get("criteria", {})):
            raise ValueError(f"{feature_id} cache has no canonical Jev yes/no template")


def generate_pair(host: dict, pathogen: dict, client: OpenAI) -> dict:
    payload = pair_payload(host, pathogen)
    checkpoint = profile_checkpoint(host, pathogen)
    if checkpoint.exists() and not REFRESH_CACHE:
        cached = json.loads(checkpoint.read_text(encoding="utf-8"))
        prior_selectors = cached.get("input", {}).get("selectors", {})
        expected_selectors = payload["selectors"]
        if prior_selectors != expected_selectors:
            raise ValueError(f"Cache selector mismatch at {checkpoint}; inspect the checkpoint before changing it")
        validate_question_records(cached.get("result", {}).get("questions", []))
        cached["cache_status"] = "reused" if cached.get("input") == payload and cached.get("model") == MODEL and cached.get("prompt_version") == PROMPT_VERSION else "reused_stale_inputs_or_generation_version"
        return cached

    response = client.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
        ],
        text_format=PairQuestionDraft,
        reasoning={"effort": REASONING_EFFORT},
        store=False,
    )
    parsed = response.output_parsed
    if parsed is None:
        raise RuntimeError(f"No structured question profile returned for {host['label']} × {pathogen['label']}")
    validate_pair_draft(parsed)
    question_records = []
    profile_prefix = f"{slug(host['host_id']).replace('-', '_')}_{pathogen['pathogen_group']}"
    for item in parsed.questions:
        feature = item.model_dump()
        # Feature IDs are globally unique across the Cartesian product, even when two
        # independently generated pair profiles use the same short descriptive ID.
        feature["feature_id"] = f"{profile_prefix}_{feature['feature_id']}"
        feature["dimension_id"] = slug(feature["dimension"]).replace("-", "_")
        feature["category_id"] = slug(feature["category"]).replace("-", "_")
        feature.update({
            "answer_type": "noul",
            "applicability": {
                "host_traits": {"all_of": feature.pop("applicability_host_traits")},
                "pathogen_traits": {"all_of": feature.pop("applicability_pathogen_traits")},
                "pair_traits": {"all_of": feature.pop("applicability_pair_traits")},
            },
        })
        jev_template = build_feature_question(
            pathogen_name="{pathogen.accepted_name}", pathogen_id="{pathogen.id}",
            host_name="{host.accepted_name}", host_id="{host.id}", feature=feature,
            article_title="{article.title}", article_abstract="{article.abstract}",
            prompt_config=jev_prompt_catalogue,
        )
        if jev_template["questions"]["feature"].get("type") != "noul":
            raise ValueError("Jev feature template must use type='noul'")
        feature["jev_question_template"] = jev_template
        question_records.append(feature)
    result = {
        "selectors": payload["selectors"],
        "curated_traits": payload["curated_traits"],
        "profile_summary": parsed.profile_summary,
        "questions": question_records,
    }
    record = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "model": MODEL,
        "reasoning_effort": REASONING_EFFORT,
        "prompt_version": PROMPT_VERSION,
        "system_prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest(),
        "input": payload,
        "result": result,
    }
    checkpoint.write_text(json.dumps(record, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return record

api = OpenAI(max_retries=5, timeout=180.0) if os.getenv("OPENAI_API_KEY") else None
if api is None:
    print("OPENAI_API_KEY is not set. Set it, then rerun this cell to generate profiles.")
else:
    exported_count = sum(export_current_catalogue_cache(host, pathogen) for host, pathogen in matrix)
    print(f"Exported {exported_count} existing resolved profiles into cache format")
    generated = []
    for index, (host, pathogen) in enumerate(matrix, start=1):
        checkpoint = profile_checkpoint(host, pathogen)
        was_cached = checkpoint.exists() and not REFRESH_CACHE
        record = generate_pair(host, pathogen, api)
        generated.append(record)
        action = "reused checkpoint" if was_cached else "generated"
        if record.get("cache_origin") == "existing_feature_catalogue":
            action = "reused exported current catalogue"
        elif record.get("cache_status") == "reused_stale_inputs_or_generation_version":
            action += " (saved inputs/model/prompt differ; cache retained)"
        print(f"[{index}/{len(matrix)}] {host['label']} × {pathogen['label']}: {action}")
    print(f"Ready: {len(generated)} host × pathogen-class profiles")


## 5 · Review coverage and save the draft YAML

In [ ]:
if api is not None and len(generated) == len(matrix):
    pair_profiles = []
    default_exclusions = {}
    for record in generated:
        profile = dict(record["result"])
        profile["profile_version"] = DRAFT_PROFILE_VERSION
        for question in profile["questions"]:
            question["source_review_status"] = question.get("review_status", "")
            question["review_status"] = "preliminary-promoted"
            question["preliminary_use"] = True
            question["selected"] = True
        profile["selected_feature_ids"] = [question["feature_id"] for question in profile["questions"]]
        profile["default_exclusions"] = []
        profile["cache_status"] = record.get("cache_status", "generated")
        profile["generation_provenance"] = {
            "model": record.get("model"),
            "prompt_version": record.get("prompt_version"),
            "cache_origin": record.get("cache_origin", "llm_generation"),
            "source_catalogue_version": record.get("source_catalogue_version"),
            "system_prompt_sha256": record.get("system_prompt_sha256"),
            "generated_at_utc": record.get("generated_at_utc"),
            "checkpoint": str(profile_checkpoint(
                next(h for h in hosts if h["host_id"] == profile["selectors"]["host_id"]),
                next(g for g in pathogen_groups if g["pathogen_group"] == profile["selectors"]["pathogen_group"]),
            ).relative_to(PROJECT_ROOT)),
        }
        pair_profiles.append(profile)
    draft = {
        "schema_version": 1,
        "catalogue_id": feature_catalogue.get("profile_id", "v4-host-aware-feature-questions"),
        "catalogue_version": DRAFT_CATALOGUE_VERSION,
        "profile_version": DRAFT_PROFILE_VERSION,
        "parent_catalogue_version": feature_catalogue.get("catalogue_version", ""),
        "status": "preliminary_active_all_questions_promoted",
        "catalogue_status": "preliminary-active",
        "preliminary_use_policy": {
            "selection": "All inherited and generated questions in the 55 resolved profiles are selected for preliminary use.",
            "future_versions": "Create a new catalogue/profile version to remove questions or add new ones; preserve this version as an immutable snapshot.",
            "review_status": "Original review status is retained per question as provenance; preliminary use is not equivalent to domain-expert validation.",
        },
        "initial_quality_review": {
            "profile_count": len(pair_profiles),
            "profile_question_count": sum(len(profile["questions"]) for profile in pair_profiles),
            "unique_feature_id_count": len({q["feature_id"] for profile in pair_profiles for q in profile["questions"]}),
            "technical_checks": "Every cached question passed feature-id, trait-vocabulary, readable-field, and Jev noul true/false-template validation.",
            "default_exclusions": default_exclusions,
            "review_limit": "All questions are enabled for preliminary use. Original review_status values remain recorded so later versions can remove or revise questions after expert review.",
        },
        "versioning_policy": {
            "snapshot_is_immutable": True,
            "stable_identity": "Keep feature_id when the meaning is unchanged.",
            "meaning_change": "Increment question_version whenever wording, scope, criteria, or interpretation changes.",
            "addition": "Add a new feature_id and complete question record in the next catalogue version.",
            "selection": "Populate each profile's selected_feature_ids to choose questions for that profile; unselected candidates remain reviewable.",
            "prior_versions": "Preserve prior version files; create and review a new version instead of editing history.",
        },
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "generation": {
            "model": MODEL,
            "reasoning_effort": REASONING_EFFORT,
            "prompt_version": PROMPT_VERSION,
            "system_prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest(),
            "host_registry": str(HOSTS_PATH.relative_to(PROJECT_ROOT)),
            "pathogen_class_registry": str(PATHOGENS_PATH.relative_to(PROJECT_ROOT)),
            "trait_registry": str(FEATURES_PATH.relative_to(PROJECT_ROOT)),
            "relationship_context": RELATIONSHIP_CONTEXT,
        },
        "matrix": {
            "host_count": len(hosts),
            "pathogen_class_count": len(pathogen_groups),
            "profile_count": len(pair_profiles),
            "hosts": hosts,
            "pathogen_classes": pathogen_groups,
        },
        "allowed_traits": {axis: sorted(values) for axis, values in allowed_traits.items()},
        "profiles": pair_profiles,
    }
    if DRAFT_ALREADY_EXISTS:
        draft = yaml.safe_load(DRAFT_PATH.read_text(encoding="utf-8"))
        print("Reusing the existing immutable draft; create a new version to change its contents.")
    else:
        DRAFT_PATH.write_text(yaml.safe_dump(draft, sort_keys=False, allow_unicode=True), encoding="utf-8")
    summary = pd.DataFrame([
        {"host": p["selectors"]["host_label"], "pathogen_class": p["selectors"]["pathogen_class_label"],
         "questions": len(p["questions"]), "pair_profile": p["selectors"]["host_id"] + " × " + p["selectors"]["pathogen_group"]}
        for p in pair_profiles
    ])
    print(f"Draft written: {DRAFT_PATH}")
    display(summary)
else:
    print("Generate all profiles in Step 4 before saving the draft catalogue.")


## Review checklist

- Feature IDs are automatically namespaced by host taxon and pathogen class so they remain unique across the complete Cartesian product.
- Confirm each pair profile has the intended host and pathogen class, and that aliases were deduplicated correctly.
- Keep only features that characterize how the pathogen causes disease in the selected host or illuminate that pathogenic relationship. Remove generic host/pathogen facts and claims unrelated to pathogenicity.
- Keep the overall disease-causation yes/no gate separate; features should characterize distinct pathogenic aspects, not repeat that gate.
- Remove redundant, speculative, bundled, or abstract-unobservable claims; revise `question`, `question_claim`, and wording as needed.
- Check every scope and applicability trait. Questions should have the narrowest valid scope, and a trait restriction should match the biological concepts it selects.
- Review whether questions should be shared across classes or hosts. This draft intentionally preserves one full profile per matrix cell rather than merging similar text automatically.
- Keep a catalogue version and profile version for every reviewed snapshot. Preserve prior files. Keep `feature_id` when meaning is stable; increment `question_version` when wording, scope, or interpretation changes.
- Choose the profile questions by editing `selected_feature_ids`; add a new question as a complete record with a new stable ID. Record the editor and review decision before integrating approved questions into `configs/features.yaml`.
- Preserve the saved prompt/model provenance and record human review decisions. LLM output is candidate design material, not a curated scientific catalogue.

## 6 · Create a new domain-expert version

For a new expert-authored version, copy the generated YAML into a version-controlled location such as `configs/feature_question_versions/`. Use the helper below to fork a previous snapshot, select candidate questions per host × class profile, and append new complete questions. It refuses to overwrite a version. Keep IDs stable when meaning is unchanged, increment `question_version` for edits that change meaning, and use a new `feature_id` for new questions. The output remains a review artifact until intentionally integrated into the runtime catalogue.

In [ ]:
import copy


def profile_key(profile: dict) -> str:
    selector = profile["selectors"]
    return f"{selector['host_id']}|{selector['pathogen_group']}"


def create_expert_version(*, source_path: Path, destination_path: Path,
                           catalogue_version: str, profile_version: str,
                           selected_feature_ids_by_profile: dict[str, list[str]],
                           additions_by_profile: dict[str, list[dict]] | None = None,
                           overrides_by_profile: dict[str, dict[str, dict]] | None = None,
                           editor: str, review_note: str) -> Path:
    """Fork a saved draft, select questions and add expert-authored records, without changing its parent."""
    if destination_path.exists():
        raise FileExistsError(f"Refusing to overwrite immutable version: {destination_path}")
    source = yaml.safe_load(source_path.read_text(encoding="utf-8"))
    versioned = copy.deepcopy(source)
    parent_version = source.get("catalogue_version", "")
    if catalogue_version == parent_version:
        raise ValueError("A new expert snapshot must use a new catalogue_version")
    if profile_version == source.get("profile_version"):
        raise ValueError("A new expert snapshot must use a new profile_version")
    additions_by_profile = additions_by_profile or {}
    overrides_by_profile = overrides_by_profile or {}
    seen_profile_keys = set()
    global_ids = {q["feature_id"] for p in versioned["profiles"] for q in p["questions"]}

    for profile in versioned["profiles"]:
        key = profile_key(profile)
        seen_profile_keys.add(key)
        questions = profile["questions"]
        known = {question["feature_id"] for question in questions}
        for question in questions:
            patch = overrides_by_profile.get(key, {}).get(question["feature_id"], {})
            if not patch:
                continue
            if "feature_id" in patch and patch["feature_id"] != question["feature_id"]:
                raise ValueError("Do not change feature_id for an existing question; add a new feature instead")
            meaning_fields = {"label", "definition", "question", "question_claim", "scope", "applicability", "answer_type"}
            meaning_changed = any(field in patch and patch[field] != question.get(field) for field in meaning_fields)
            if meaning_changed and patch.get("question_version", question.get("question_version")) == question.get("question_version"):
                raise ValueError(f"Meaning changed for {question['feature_id']}; increment question_version")
            question.update(copy.deepcopy(patch))
            if question.get("answer_type") != "noul":
                raise ValueError(f"Expert-edited feature must remain a Jev noul yes/no question: {question['feature_id']}")
            template = build_feature_question(
                pathogen_name="{pathogen.accepted_name}", pathogen_id="{pathogen.id}",
                host_name="{host.accepted_name}", host_id="{host.id}", feature=question,
                article_title="{article.title}", article_abstract="{article.abstract}",
                prompt_config=jev_prompt_catalogue,
            )
            question["jev_question_template"] = template
            question["review_status"] = "reviewed"

        for addition in additions_by_profile.get(key, []):
            added = copy.deepcopy(addition)
            required = {"feature_id", "question_version", "dimension", "dimension_id", "category", "category_id",
                        "label", "definition", "question", "question_claim", "scope", "answer_type", "applicability"}
            missing = required - set(added)
            if missing:
                raise ValueError(f"Expert addition for {key} is missing fields: {sorted(missing)}")
            if added["feature_id"] in known or added["feature_id"] in global_ids:
                raise ValueError(f"Feature ID already exists in this catalogue: {added['feature_id']}")
            if added["answer_type"] != "noul":
                raise ValueError(f"Expert addition must remain a Jev noul yes/no question: {added['feature_id']}")
            template = build_feature_question(
                pathogen_name="{pathogen.accepted_name}", pathogen_id="{pathogen.id}",
                host_name="{host.accepted_name}", host_id="{host.id}", feature=added,
                article_title="{article.title}", article_abstract="{article.abstract}",
                prompt_config=jev_prompt_catalogue,
            )
            if template["questions"]["feature"].get("type") != "noul":
                raise ValueError(f"Jev template is not yes/no for {added['feature_id']}")
            added["jev_question_template"] = template
            added["review_status"] = "reviewed"
            added["selected"] = False  # Inclusion is explicit in selected_feature_ids_by_profile.
            questions.append(added)
            known.add(added["feature_id"])
            global_ids.add(added["feature_id"])

        selected = list(dict.fromkeys(selected_feature_ids_by_profile.get(key, [])))
        unknown = set(selected) - known
        if unknown:
            raise ValueError(f"Selected IDs missing from profile {key}: {sorted(unknown)}")
        profile["selected_feature_ids"] = selected
        for question in questions:
            question["selected"] = question["feature_id"] in selected
            if question["selected"] and question.get("review_status") == "curated-draft":
                question["review_status"] = "reviewed"

    unknown_profiles = (set(selected_feature_ids_by_profile) | set(additions_by_profile) | set(overrides_by_profile)) - seen_profile_keys
    if unknown_profiles:
        raise ValueError(f"Unknown host × pathogen-class profile keys: {sorted(unknown_profiles)}")

    versioned["parent_catalogue_version"] = parent_version
    versioned["catalogue_version"] = catalogue_version
    versioned["profile_version"] = profile_version
    versioned["status"] = "domain_expert_review_version"
    versioned["versioning_policy"]["editor"] = editor
    versioned["versioning_policy"]["review_note"] = review_note
    versioned["versioning_policy"]["created_from"] = str(source_path)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    destination_path.write_text(yaml.safe_dump(versioned, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return destination_path

# Example only: fill these after review, then uncomment and provide the expert's name/notes.
# PROFILE_SELECTIONS = {"ncbi:4530|viruses": ["ncbi_4530_viruses_example_feature"]}
# EXPERT_ADDITIONS = {"ncbi:4530|viruses": [{... complete question record ...}]}
# EXPERT_OVERRIDES = {"ncbi:4530|viruses": {"existing_feature_id": {"question": "...", "question_version": "v2"}}}
# create_expert_version(
#     source_path=DRAFT_PATH,
#     destination_path=PROJECT_ROOT / "configs/feature_question_versions/v4-feature-catalogue-2.1.0.yaml",
#     catalogue_version="v4-feature-catalogue-2.1.0",
#     profile_version="v4-feature-profile-2.1.0",
#     selected_feature_ids_by_profile=PROFILE_SELECTIONS,
#     additions_by_profile=EXPERT_ADDITIONS,
#     overrides_by_profile=EXPERT_OVERRIDES,
#     editor="Domain expert name",
#     review_note="Summary of the review and changes.",
# )
